# Tests des fonctions de calcul de la saturation

In [1]:
import json
from datetime import date, timedelta

import pandas as pd

#from saturation_image_quali_prod import (
from saturation_image_quali import (
    filter_sessions_duration,
    get_sampled_state_poc,
    to_sampled_state_grp,
    to_state_grp_d,
    to_state_grp_h,
    to_state_poc_d,
)

SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
day = date(2026,7, 14)
#date_calcul = "2026-07-05"
date_calcul = "2026-07-14"
date_file = date_calcul.replace("-", "")

data_quali = "../data/"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for POC and stations."""
    e5_str = pd.read_csv("../data_DMR_e2_e3/e5_05-07-2026.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]


In [3]:
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
statuses_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")

In [4]:
sessions = pd.read_csv('../data_test/donnees_sessions_FRHPCPNF050462_05-07-2026.csv')[['start', 'end', 'id_pdc_itinerance']]
sessions['start'] = pd.to_datetime(sessions['start'])
sessions['end'] = pd.to_datetime(sessions['end'])
statuses = pd.DataFrame({ID_POC:[], "horodatage":[], "etat_pdc":[], "occupation_pdc":[]})
statuses['horodatage'] = pd.to_datetime(statuses['horodatage'], utc=True)
# statuses = pd.DataFrame()
statics = pd.DataFrame({ID_POC:['FRHPCENF050462001', 'FRHPCENF050462002', 'FRHPCENF050462004' ], ID_STATION:['FRHPCPNF050462']*3})

samples_per_day = SAMPLES #72 #288

e2_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][0]
e2_pdc = pd.DataFrame(json.loads(e2_str))

e3_str = pd.read_csv("../data_DMR_e2_e3/e2_e3_05-07-2026.csv")["extras"][1]
e3_station = pd.DataFrame(json.loads(e3_str))

e5_statics = read_statics(day, MIN_POWER)

# FRHPCENF050462001
e2_pdc = e2_pdc[e2_pdc[ID_POC].str[:14] == 'FRHPCENF050462']
e3_station = e3_station[e3_station[ID_STATION] == 'FRHPCPNF050462']


In [5]:
#e5_statics[e5_statics[ID_STATION] == 'FRHPCPNF080266TOTEM']

## test local

In [6]:
samples_per_day = 288
#sampled_state_poc = get_sampled_state_poc(day, samples_per_day, sessions, statuses)

In [7]:
#sampled_state_poc[sampled_state_poc[ID_POC] == 'FRHPCENF050462001']

In [8]:
#state_poc_d = to_state_poc_d(sampled_state_poc, samples_per_day)

In [9]:
#state_poc_d, e2_pdc

In [10]:
#sample_state_station = to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
#state_station_h = to_state_grp_h(sample_state_station, ID_STATION, SAMPLES, SATURE_H)
#state_station_d = to_state_grp_d(state_station_h, ID_STATION)

In [11]:
#state_station_d

In [12]:
#e3_station

## test global

In [13]:
#sessions_s3[sessions_s3[ID_POC] == 'FRHPCENF050462002']

In [14]:
samples_per_day = 288

min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)
sessions = filter_sessions_duration(sessions_s3, min_duration=min_duration, max_duration=max_duration)

sampled_state_poc_g = get_sampled_state_poc(day, samples_per_day, sessions, statuses_s3)

In [15]:
"""
from datetime import timedelta

from saturation_image_quali import to_sampled_sessions, to_sampled_statuses

MAX_SESSION_DURATION_HOURS: float = 10
sessions = sessions_s3.copy()
statuses = statuses_s3.copy()
min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)
timestamp = pd.Timestamp(day.isoformat() + "T00:00:00+00:00")
pocs_with_sessions = pd.Series(sessions[ID_POC].unique())
pocs_with_statuses = pd.Series(statuses[ID_POC].unique())
all_pocs = pd.concat([pocs_with_sessions, pocs_with_statuses]).drop_duplicates().reset_index(drop=True)

attributes_statuses = [ID_POC, "horodatage", "etat_pdc", "occupation_pdc"]
statuses = statuses[attributes_statuses].copy()
statuses["horodatage"] = statuses["horodatage"].astype("datetime64[s, UTC]")

attributes_sessions = [ID_POC, "start", "end"]
sessions = sessions[attributes_sessions].copy()
sessions["start"] = sessions["start"].astype("datetime64[s, UTC]")
sessions["end"] = sessions["end"].astype("datetime64[s, UTC]")

init_start_statuses = pd.DataFrame(
    {
        "horodatage": [timestamp + pd.Timedelta(days=-1)] * len(all_pocs),
        "etat_pdc": ["en_service"] * len(all_pocs),
        "occupation_pdc": ["libre"] * len(all_pocs),
        "id_pdc_itinerance": all_pocs,
    }
)
init_end_statuses = pd.DataFrame(
        {
            "horodatage": [timestamp + pd.Timedelta(days=1)] * len(all_pocs),
            "etat_pdc": ["en_service"] * len(all_pocs),
            "occupation_pdc": ["libre"] * len(all_pocs),
            "id_pdc_itinerance": all_pocs,
        }
    )
init_statuses = pd.concat([init_start_statuses, init_end_statuses])

init_statuses["horodatage"] = pd.to_datetime(init_statuses["horodatage"], utc=True)
sampled_statuses = to_sampled_statuses(
    statuses, init_statuses, timestamp, samples_per_day, min_duration=min_duration
)

init_sessions = pd.DataFrame(
    {
        "start": [timestamp + pd.Timedelta(hours=-2)] * len(all_pocs),
        "end": [timestamp + pd.Timedelta(hours=-1)] * len(all_pocs),
        "id_pdc_itinerance": all_pocs,
    }
)
sampled_sessions = to_sampled_sessions(
    sessions,
    init_sessions,
    timestamp,
    samples_per_day,
    min_duration=min_duration,
    max_duration=max_duration,
)
"""

'\nfrom datetime import timedelta\n\nfrom saturation_image_quali import to_sampled_sessions, to_sampled_statuses\n\nMAX_SESSION_DURATION_HOURS: float = 10\nsessions = sessions_s3.copy()\nstatuses = statuses_s3.copy()\nmin_duration = timedelta(minutes=24 * 60 / samples_per_day)\nmax_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)\ntimestamp = pd.Timestamp(day.isoformat() + "T00:00:00+00:00")\npocs_with_sessions = pd.Series(sessions[ID_POC].unique())\npocs_with_statuses = pd.Series(statuses[ID_POC].unique())\nall_pocs = pd.concat([pocs_with_sessions, pocs_with_statuses]).drop_duplicates().reset_index(drop=True)\n\nattributes_statuses = [ID_POC, "horodatage", "etat_pdc", "occupation_pdc"]\nstatuses = statuses[attributes_statuses].copy()\nstatuses["horodatage"] = statuses["horodatage"].astype("datetime64[s, UTC]")\n\nattributes_sessions = [ID_POC, "start", "end"]\nsessions = sessions[attributes_sessions].copy()\nsessions["start"] = sessions["start"].astype("datetime64[s, UTC]")\nses

In [16]:
#sampled_sessions[sampled_sessions[ID_POC] == 'FRA79E12346905331']

In [17]:
#sampled_statuses[sampled_statuses[ID_POC] == 'FRA79E12346905331'][100:150]

In [18]:
#sampled_state_poc_g[sampled_state_poc_g['pseudo_occupe'] >0]

In [19]:
state_poc_d_g = to_state_poc_d(sampled_state_poc_g, samples_per_day)

In [20]:
#state_poc_d_g[state_poc_d_g[ID_POC].str[:14] == 'FRHPCENF050462']

In [21]:
state_poc_d_g[state_poc_d_g['pseudo_libre'] > 0]

,id_pdc_itinerance,occupe,hors_service,libre,pseudo_libre,pseudo_occupe
0,FR3R3E10000849901,55.0,195.0,1190.0,55.0,0.0
29,FRA79E12346973541,50.0,0.0,1390.0,5.0,0.0
75,FRA79E12347181491,45.0,0.0,1395.0,5.0,0.0
80,FRA79E12347315251,335.0,0.0,1105.0,80.0,5.0
81,FRA79E12347315252,700.0,0.0,740.0,15.0,0.0
...,...,...,...,...,...,...
45543,FRZUNEFR5701ER03,20.0,0.0,1420.0,20.0,0.0
45550,FRZUNEFR5702ER02,25.0,20.0,1395.0,15.0,0.0
45556,FRZUNEFR8801ER01,60.0,0.0,1380.0,20.0,550.0
45558,FRZUNEFR8801ER03,70.0,0.0,1370.0,35.0,0.0


In [22]:
state_poc_d_g[state_poc_d_g['pseudo_occupe'] > 0]

,id_pdc_itinerance,occupe,hors_service,libre,pseudo_libre,pseudo_occupe
1,FR3R3E10001456611,90.0,35.0,1315.0,0.0,5.0
22,FRA79E12346973253,155.0,45.0,1240.0,0.0,5.0
32,FRA79E12346974443,80.0,0.0,1360.0,0.0,55.0
65,FRA79E12347058702,190.0,0.0,1250.0,0.0,5.0
70,FRA79E12347133852,240.0,0.0,1200.0,0.0,30.0
...,...,...,...,...,...,...
45549,FRZUNEFR5702ER01,10.0,0.0,1430.0,0.0,385.0
45551,FRZUNEFR5702ER05,35.0,0.0,1405.0,0.0,730.0
45556,FRZUNEFR8801ER01,60.0,0.0,1380.0,20.0,550.0
45557,FRZUNEFR8801ER02,105.0,0.0,1335.0,0.0,90.0


In [23]:
# sampled_state_poc_g[sampled_state_poc_g[ID_POC] == 'FRA79E12346905331'][0:100]

In [24]:
#sessions_s3[sessions_s3[ID_POC] == 'FRA79E12346905331']

In [25]:
#statuses_s3[statuses_s3[ID_POC] == 'FRA79E12346905331']

In [26]:
sampled_state_station_g = to_sampled_state_grp(sampled_state_poc_g, e5_statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO)
print(len(sampled_state_station_g))

438336


In [27]:
def filter_sampled_state_poc(
    sampled_state_poc: pd.DataFrame, statics: pd.DataFrame
) -> pd.DataFrame:
    """Filter statuses and sessions with statics data."""
    filtered = sampled_state_poc[sampled_state_poc[ID_POC].isin(statics[ID_POC])].copy()
    return filtered

chunk_size = 200
codes, _ = pd.factorize(e5_statics[ID_STATION])
e5_statics["chunk"] = codes // chunk_size
chunks = e5_statics.groupby("chunk")
print(len(chunks))

futures = [
    to_sampled_state_grp(
        sampled_state_poc_g[sampled_state_poc_g[ID_POC].isin(chunk[ID_POC])],
        chunk,
        ID_STATION,
        SATURATION_RATIO,
        OVERLOAD_RATIO,
    )  # type: ignore[call-overload]
    for _, chunk in chunks
]

sampled_state_station_g = pd.concat(
    [future for future in futures], ignore_index=True
)
print(len(sampled_state_station_g))

15
438336


In [28]:
state_station_h_g = to_state_grp_h(sampled_state_station_g, ID_STATION, SAMPLES, SATURE_H)

state_station_d_g = to_state_grp_d(sampled_state_station_g, ID_STATION, SAMPLES)

In [29]:
sampled_state_station_g

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
0,FRALLPGO000020,2026-07-14 00:00:00+00:00,0,0,6,0,6,False,True,False,False,False,False,2
1,FRALLPGO000020,2026-07-14 00:05:00+00:00,0,0,6,0,6,False,True,False,False,False,False,2
2,FRALLPGO000020,2026-07-14 00:10:00+00:00,0,0,6,0,6,False,True,False,False,False,False,2
3,FRALLPGO000020,2026-07-14 00:15:00+00:00,0,0,6,0,6,False,True,False,False,False,False,2
4,FRALLPGO000020,2026-07-14 00:20:00+00:00,0,0,6,0,6,False,True,False,False,False,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
438331,FRZUNP1602450077204251630,2026-07-14 23:35:00+00:00,0,0,1,0,2,False,True,False,False,False,False,2
438332,FRZUNP1602450077204251630,2026-07-14 23:40:00+00:00,0,0,1,0,2,False,True,False,False,False,False,2
438333,FRZUNP1602450077204251630,2026-07-14 23:45:00+00:00,0,0,1,0,2,False,True,False,False,False,False,2
438334,FRZUNP1602450077204251630,2026-07-14 23:50:00+00:00,0,0,1,0,2,False,True,False,False,False,False,2


In [30]:
sampled_state_station_g[sampled_state_station_g[ID_STATION] == 'FRHPCPNF080266TOTEM']

,id_station_itinerance,periode,occupe,hors_service,libre,pleine_utilisation,nb_pdc,hs,inactif,pu,sature,surcharge,actif,state
410688,FRHPCPNF080266TOTEM,2026-07-14 00:00:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410689,FRHPCPNF080266TOTEM,2026-07-14 00:05:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410690,FRHPCPNF080266TOTEM,2026-07-14 00:10:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410691,FRHPCPNF080266TOTEM,2026-07-14 00:15:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410692,FRHPCPNF080266TOTEM,2026-07-14 00:20:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410971,FRHPCPNF080266TOTEM,2026-07-14 23:35:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410972,FRHPCPNF080266TOTEM,2026-07-14 23:40:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410973,FRHPCPNF080266TOTEM,2026-07-14 23:45:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2
410974,FRHPCPNF080266TOTEM,2026-07-14 23:50:00+00:00,0,0,2,0,2,False,True,False,False,False,False,2


In [31]:
state_station_h_g

,id_station_itinerance,periode_d,periode_h,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_h,surcharge_h
0,FRALDPFR00916,2026-07-14,0,1,0.0,60.0,0.0,0.0,0.0,0.0,False,False
1,FRALDPFR00916,2026-07-14,1,1,0.0,60.0,0.0,0.0,0.0,0.0,False,False
2,FRALDPFR00916,2026-07-14,2,1,0.0,60.0,0.0,0.0,0.0,0.0,False,False
3,FRALDPFR00916,2026-07-14,3,1,0.0,60.0,0.0,0.0,0.0,0.0,False,False
4,FRALDPFR00916,2026-07-14,4,1,0.0,60.0,0.0,0.0,0.0,0.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
36523,FRZUNP7329346578064027187,2026-07-14,19,2,0.0,60.0,0.0,0.0,0.0,0.0,False,False
36524,FRZUNP7329346578064027187,2026-07-14,20,2,0.0,60.0,0.0,0.0,0.0,0.0,False,False
36525,FRZUNP7329346578064027187,2026-07-14,21,2,0.0,60.0,0.0,0.0,0.0,0.0,False,False
36526,FRZUNP7329346578064027187,2026-07-14,22,2,0.0,60.0,0.0,0.0,0.0,0.0,False,False


In [32]:
state_station_d_g

,id_station_itinerance,nb_pdc,nb_h,hs,inactif,sature_cum,surcharge,actif
0,FRALDPFR00916,1,1440.0,0.0,1425.0,15.0,0.0,0.0
1,FRALDPFR00950,1,1440.0,0.0,1400.0,40.0,0.0,0.0
2,FRALLPGO000007,6,1440.0,0.0,1010.0,0.0,0.0,430.0
3,FRALLPGO000013,10,1440.0,0.0,520.0,0.0,0.0,920.0
4,FRALLPGO000014,2,1440.0,0.0,1140.0,65.0,0.0,235.0
...,...,...,...,...,...,...,...,...
1517,FRZUNP2737850063890267871,2,1440.0,0.0,1415.0,25.0,0.0,0.0
1518,FRZUNP2923250052026028059,1,1440.0,285.0,1110.0,45.0,0.0,0.0
1519,FRZUNP5724050077640808316,3,1440.0,0.0,1405.0,35.0,0.0,0.0
1520,FRZUNP5909246146811129817,1,1440.0,0.0,1395.0,45.0,0.0,0.0


In [33]:
state_station_d_g[state_station_d_g[ID_STATION] == 'FRHPCPNF080266TOTEM']

,id_station_itinerance,nb_pdc,nb_h,hs,inactif,sature_cum,surcharge,actif
808,FRHPCPNF080266TOTEM,2,1440.0,0.0,940.0,175.0,0.0,325.0


In [34]:
state_station_d_g[state_station_d_g["sature_cum"] >= 120]

,id_station_itinerance,nb_pdc,nb_h,hs,inactif,sature_cum,surcharge,actif
26,FRALLPGO000275,2,1440.0,0.0,1260.0,180.0,0.0,0.0
35,FRALLPGO000331,4,1440.0,0.0,1180.0,260.0,0.0,0.0
125,FRALLPGO000692,4,1440.0,0.0,1095.0,120.0,0.0,225.0
144,FRALLPGO000794,4,1440.0,0.0,1295.0,145.0,0.0,0.0
165,FRALLPGO000953,2,1440.0,0.0,815.0,180.0,0.0,445.0
...,...,...,...,...,...,...,...,...
1456,FRVIAP102106,2,1440.0,5.0,780.0,305.0,0.0,350.0
1458,FRVIAP103103,3,1440.0,0.0,700.0,305.0,0.0,435.0
1460,FRVIAP103110,2,1440.0,0.0,1045.0,395.0,0.0,0.0
1461,FRVIAP103112,1,1440.0,0.0,1095.0,345.0,0.0,0.0


In [35]:
static = pd.DataFrame({'station' : ['s1']*3+['s2']*5+['s3']*3+['s4']*4+['s5']*5,
                      'pdc': ['p'+str(i) for i in range(20)]})
#static

In [36]:
chunk = 2
station_to_chunk = {
    station: i // chunk
    for i, station in enumerate(static["station"].drop_duplicates())
}

static["chunk"] = static["station"].map(station_to_chunk)
#for chunk_id, chunk in static.groupby("chunk", sort=True):
#    print(chunk_id)
#    print(chunk)
#static

In [37]:
chunk_size = 3
'''station_to_chunk = {
    station: i // chunk_size
    for i, station in enumerate(static["station"].drop_duplicates())
}

static["chunk"] = static["station"].map(station_to_chunk)
chunks = static.groupby("chunk", sort=True)
'''
codes, _ = pd.factorize(static["station"])
static["chunk"] = codes//chunk_size
chunks = static.groupby("chunk")
#for chunk_id, chunk in chunks:
    #print(chunk_id)
    #print(chunk)
futures = [
    chunk for _ , chunk in chunks
    #to_sampled_state_grp(sampled_state_poc, statics, ID_STATION, SATURATION_RATIO, OVERLOAD_RATIO) for _ , chunk in chunks
] 
static_new = pd.concat(
        [future for future in futures], ignore_index=True
    )
#static_new